In [2]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [3]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

In [4]:
# Q1 solution
len(documents)

72

In [5]:
# Q2 solution
from minsearch import Index

def build_index(documents):
    index = Index(
        text_fields=['content'],
        keyword_fields=['filename']
    )
    index.fit(documents)
    return index


idx = build_index(documents)

In [6]:
search_res = idx.search("How does the agentic loop keep calling the model until it stops?")

In [74]:
search_res[0]

{'content': '# The Agentic Loop\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=ePlQUcTPPjw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we did function calling by hand. We sent a\nmessage and got back a function call. We ran it, sent the result back,\nand got the answer.\n\nThat works for one function call. It breaks down when the model wants\nto search several times, or when the first search misses the answer.\nWe don\'t know in advance how many calls the model will want. So we\nneed a loop that keeps calling the model and running tools until it\'s\ndone. An agent is exactly that.\n\n## Anatomy of an agent\n\nWith the LLM in the driver\'s seat, we have an agent. It\'s an AI\nassistant whose goal is to help the user.\n\nAn agent has three parts:\n\n- Instructions, the role and behavior we want. We pass this as the\n  `developer` message. The better the instructions, the better the\n  agent helps.\n- Tools, the functions the agent can call to carry 

In [18]:
q2_filename=search_res[0]["filename"]
print(q2_filename)

01-agentic-rag/lessons/14-agentic-loop.md


In [11]:
#Q3 solution

from rag_helper import RAGBase

In [14]:
search_res[0].keys()

dict_keys(['content', 'filename'])

In [ ]:
class RAGHW1(RAGBase):

    # using filename as an alternative of self.course

    def search(self, query, num_results=3):
        boost_dict = {'content': 3.0, 'filename': 0.1}
        filter_dict = {'filename': self.course}

        return self.index.search(
            query,
            num_results=num_results,
            boost_dict=boost_dict,
            filter_dict=filter_dict
        )
    
    def build_context(self, search_results):
        lines = []

        for doc in search_results:
            lines.append(doc['content'])
            lines.append('')

        return '\n'.join(lines).strip()
    
    def llm(self, prompt):
        input_messages = [
            {'role': 'developer', 'content': self.instructions},
            {'role': 'user', 'content': prompt}
        ]

        response = self.llm_client.responses.create(
            model=self.model,
            input=input_messages
        )

        return response.output_text, response.usage



In [76]:
from openai import OpenAI
from dotenv import load_dotenv

openai_client = OpenAI()
load_dotenv()


True

In [77]:
raghw = RAGHW1(index=idx, llm_client=openai_client, course=q2_filename)

In [78]:
q3 = raghw.rag("How does the agentic loop keep calling the model until it stops?")

In [79]:
q3[1]

ResponseUsage(input_tokens=2326, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=126, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=2452)

In [59]:
#Q4

In [60]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

In [61]:
len(chunks)

295

In [65]:
chunks[0]

{'start': 0,
 'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phon

In [66]:
chunks[1]

{'start': 1000,
 'content': 'the next\nword based on what you typed so far.\n\nA large language model does the same thing, but at a much larger scale.\nIt has billions of parameters and is trained on most of the text on the\ninternet. When it predicts the next word, it feels like you\'re talking\nto an intelligent being. It understands what you ask and gives\nmeaningful answers.\n\nIn this course, we treat LLMs as black boxes. We won\'t look inside or\ncover the theory, and we won\'t host a model ourselves. We use an LLM\nprovider and call it over an API. For us, an LLM is a box: text goes in,\ntext comes out.\n\nBut LLMs have limitations:\n\n- Knowledge cutoff: they only know what was in their training data.\n  If you ask about something that happened after training, they won\'t\n  know - or worse, they\'ll make something up.\n- No access to your data: they can\'t see your documents, databases,\n  or internal systems unless you provide that information.\n- Hallucinations: they sometim

In [68]:
idx_chunked = build_index(chunks)

In [69]:
#Q5 
raghw = RAGHW1(index=idx_chunked, llm_client=openai_client, course=q2_filename)
q5 = raghw.rag("How does the agentic loop keep calling the model until it stops?")

In [71]:
q5[1]

ResponseUsage(input_tokens=1436, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=121, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=1557)

In [72]:
1436/(2326+1792)

0.348712967459932

In [73]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [86]:
def search_agent(query: str) -> dict[str, str]:
    """
    Searches the course description pages for entries matching the given query.
    """

    boost_dict = {'content': 3.0, 'filename': 0.1}

    return idx_chunked.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        #filter_dict=filter_dict
    )

In [87]:
agent_tools = Tools()
agent_tools.add_tool(search_agent)
agent_tools.get_tools()

 

[{'type': 'function',
  'name': 'search_agent',
  'description': 'Searches the course description pages for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [90]:
agent_instructions= """
You're a course teaching assistant. 
Answer the student's question using the search tool (search_agent). 
Make multiple searches with different keywords before answering.

If you want to look up information, use the search_agent function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

The question has to be about the course, LLMs, AI, or related topics, off-topic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using the search_agent function, don't do it yourself. Only use the 
facts from the document database.

At the end, ask if there are other areas that the user wants to explore.
""".strip()


In [91]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=agent_instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)


result = runner.loop(
    prompt="How does the agentic loop work, and how is it different from plain RAG?",
    callback=callback,
)

-> Response received


-> Response received


-> Response received
